# Day 9: Production Patterns — FastAPI & Containerisation

**Learning Goals:**
- Package the chains, agents, and tools built across Days 1–8 into importable Python modules (`src/`)
- Build a production-ready FastAPI application exposing the AI pipelines via REST endpoints
- Add streaming (Server-Sent Events) for real-time token output
- Containerise the app with Docker and docker-compose
- Smoke-test every endpoint from this notebook

**Time:** 3–4 hours

---

## What We're Building

```
src/
├── utils/llm.py          ← Shared LLM + embeddings factory
├── chains/
│   ├── chat.py           ← Conversational chain (Days 1-2)
│   └── rag.py            ← RAG chain with citations (Days 3-4)
├── tools/
│   └── custom_tools.py   ← calculator, word_counter, get_weather, text_summarizer (Day 5)
├── agents/
│   └── react_agent.py    ← ReAct / tool-calling agent (Day 5)
├── memory/
│   └── vector_store.py   ← ChromaDB helpers (Days 3-4)
└── api/
    ├── models.py          ← Pydantic request/response schemas
    ├── dependencies.py    ← FastAPI dependency injection
    └── main.py            ← FastAPI app: /health /chat /chat/stream /rag/query /agent/run
```

---

## API Endpoints

| Method | Path | Description |
|--------|------|-------------|
| GET | `/health` | Liveness check |
| POST | `/chat` | Conversational LLM chain |
| POST | `/chat/stream` | Same, with SSE token streaming |
| POST | `/rag/query` | RAG pipeline with source citations |
| POST | `/agent/run` | ReAct agent with tools |

## Part 1: Environment Setup

Load API keys and verify the project is on `sys.path` so imports from `src/` work.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# ── Make sure the project root is on sys.path so 'src.' imports work ─────────
project_root = Path().resolve().parent  # notebooks/ → project root
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
LANGCHAIN_API_KEY  = os.getenv("LANGCHAIN_API_KEY")

print("✅ OpenRouter key loaded" if OPENROUTER_API_KEY else "⚠️  Missing OPENROUTER_API_KEY — add it to .env")
print("✅ LangSmith key loaded"  if LANGCHAIN_API_KEY  else "ℹ️  LangSmith tracing disabled")
print(f"📂 Project root on path: {project_root}")

## Part 2: Smoke-test the `src/` Modules

Verify every module imports cleanly and the key classes/functions are reachable — before we spin up a server.

In [ ]:
### 2a. LLM + Embeddings factory
from src.utils.llm import get_llm, get_embeddings

llm = get_llm(temperature=0.7)
print(f"✅ LLM: {llm.model_name}")

embeddings = get_embeddings()
print(f"✅ Embeddings: {embeddings.model}")

In [ ]:
### 2b. Custom Tools (from Day 5)
from src.tools.custom_tools import ALL_TOOLS

print("✅ Custom tools loaded:")
for t in ALL_TOOLS:
    print(f"  🔧 {t.name}: {t.description[:70].strip()}…")

# Quick inline tests (no LLM call needed)
from src.tools.custom_tools import calculator, word_counter, get_weather

print("\n--- calculator ---")
print(calculator.invoke("42 * 7"))

print("\n--- word_counter ---")
print(word_counter.invoke("The quick brown fox jumps over the lazy dog."))

print("\n--- get_weather ---")
print(get_weather.invoke("Tokyo"))

In [ ]:
### 2c. Chat Chain (from Days 1-2)
from src.chains.chat import build_chat_chain

chat_chain = build_chat_chain()
print("✅ Chat chain built successfully")
print(f"   Type: {type(chat_chain).__name__}")

# Test a simple invocation
answer = chat_chain.invoke({"question": "What is LangChain in one sentence?", "history": []})
print(f"\n🤖 Answer: {answer}")

In [ ]:
### 2d. ReAct Agent (from Day 5)
from src.agents.react_agent import build_react_agent, run_agent

agent = build_react_agent(verbose=False)
print("✅ ReAct agent built successfully")
print(f"   Tools available: {[t.name for t in agent.tools]}")

# Quick test — use the calculator tool
result = run_agent("What is 144 divided by 12, then multiplied by 7?", agent=agent)
print(f"\n🤖 Output: {result['output']}")
if result["steps"]:
    print(f"   Steps  : {len(result['steps'])} tool call(s)")
    for step in result["steps"]:
        print(f"     └ {step['tool']}({step['tool_input']}) → {step['observation'][:60]}")

## Part 3: FastAPI Application Architecture

Before starting the server, let's inspect the app to confirm all routes are registered.

In [ ]:
from src.api.main import app

print(f"📱 App title  : {app.title}")
print(f"🔢 App version: {app.version}")
print("\n📍 Registered routes:")
for route in app.routes:
    if hasattr(route, "methods"):
        methods = ",".join(route.methods)
        print(f"  [{methods:6}] {route.path}")
    else:
        print(f"  [MOUNT ] {route.path}")

## Part 4: Start the FastAPI Server

We launch `uvicorn` as a background subprocess so we can call the API from subsequent cells.

> **Note:** The server runs on `http://localhost:8000`. Running this cell again will restart it.

In [ ]:
import subprocess
import time
import sys
import os

# Kill any previous server instance on port 8000
subprocess.run(["pkill", "-f", "uvicorn src.api.main"], stderr=subprocess.DEVNULL)
time.sleep(1)

# Launch uvicorn in the background
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "src.api.main:app", "--port", "8000", "--log-level", "warning"],
    cwd=str(project_root),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env={**os.environ, "PYTHONPATH": str(project_root)},
)

print(f"🚀 Server starting (PID {server_proc.pid})…")

# Poll the health endpoint until the server is ready (max 30s)
import urllib.request
import urllib.error

for attempt in range(30):
    time.sleep(1)
    try:
        with urllib.request.urlopen("http://localhost:8000/health", timeout=2) as resp:
            if resp.status == 200:
                print(f"✅ Server ready after {attempt + 1}s — http://localhost:8000")
                print("   Swagger UI: http://localhost:8000/docs")
                break
    except (urllib.error.URLError, ConnectionRefusedError):
        pass
else:
    print("⚠️  Server did not become ready in 30s. Check logs:")
    print(server_proc.stderr.read(2000).decode())

## Part 5: Testing the API Endpoints

We use `httpx` (sync client) to call each endpoint and inspect the responses.

Install it if needed:
```bash
pip install httpx
```

In [ ]:
import httpx, json

BASE_URL = "http://localhost:8000"
client   = httpx.Client(base_url=BASE_URL, timeout=60.0)

# ── 5a. Health check ──────────────────────────────────────────────────────────
print("=" * 50)
print("5a. GET /health")
print("=" * 50)
resp = client.get("/health")
print(f"Status : {resp.status_code}")
print(f"Body   : {resp.json()}")

In [ ]:
# ── 5b. POST /chat ────────────────────────────────────────────────────────────
print("=" * 50)
print("5b. POST /chat")
print("=" * 50)

payload = {
    "question": "What is RAG in AI, in two sentences?",
    "history": [],
}
resp = client.post("/chat", json=payload)
print(f"Status : {resp.status_code}")
data   = resp.json()
print(f"Answer : {data['answer']}")
print(f"Model  : {data['model']}")

In [ ]:
# ── 5c. POST /chat — multi-turn with history ──────────────────────────────────
print("=" * 50)
print("5c. POST /chat — multi-turn conversation")
print("=" * 50)

payload = {
    "question": "Can you give me a concrete example of it?",
    "history": [
        {"role": "user",      "content": "What is RAG in AI, in two sentences?"},
        {"role": "assistant", "content": data["answer"]},  # from previous cell
    ],
}
resp = client.post("/chat", json=payload)
print(f"Status : {resp.status_code}")
print(f"Answer : {resp.json()['answer']}")

In [ ]:
# ── 5d. POST /chat/stream — Server-Sent Events ────────────────────────────────
print("=" * 50)
print("5d. POST /chat/stream (SSE streaming)")
print("=" * 50)

payload = {"question": "Name three use-cases for vector databases.", "history": []}

print("Streaming tokens: ", end="", flush=True)
token_count = 0

with client.stream("POST", "/chat/stream", json=payload) as response:
    for line in response.iter_lines():
        if line.startswith("data: "):
            token = line[6:]          # strip "data: " prefix
            if token == "[DONE]":
                break
            if token.startswith("[ERROR]"):
                print(f"\n⚠️  {token}")
                break
            print(token, end="", flush=True)
            token_count += 1

print(f"\n\n✅ Received {token_count} token chunk(s)")

In [ ]:
# ── 5e. POST /rag/query ───────────────────────────────────────────────────────
print("=" * 50)
print("5e. POST /rag/query")
print("=" * 50)
print("ℹ️  Requires a populated ChromaDB collection.")
print("   If you see a 503 error, re-run the Day 3 or Day 4 notebook first.\n")

payload = {"question": "What is RAG and why is it useful?", "k": 3}
resp    = client.post("/rag/query", json=payload)

print(f"Status  : {resp.status_code}")
if resp.status_code == 200:
    body = resp.json()
    print(f"Answer  : {body['answer'][:300]}…" if len(body["answer"]) > 300 else f"Answer  : {body['answer']}")
    print(f"\n📚 Sources ({len(body['sources'])}):")
    for s in body["sources"]:
        print(f"  [{s['document_type']:12}] {s['title']}")
        print(f"              Snippet: {s['snippet'][:80]}…")
else:
    print(f"Detail  : {resp.json().get('detail', 'unknown error')}")

In [ ]:
# ── 5f. POST /agent/run ───────────────────────────────────────────────────────
print("=" * 50)
print("5f. POST /agent/run")
print("=" * 50)

payload = {
    "query": (
        "What is the square root of 625? "
        "And how many words are in the sentence 'The quick brown fox'?"
    )
}
resp = client.post("/agent/run", json=payload, timeout=120.0)

print(f"Status  : {resp.status_code}")
if resp.status_code == 200:
    body = resp.json()
    print(f"Output  : {body['output']}")
    print(f"\n🔍 Tool calls ({len(body['steps'])}):")
    for i, step in enumerate(body["steps"], 1):
        print(f"  {i}. {step['tool']}({step['tool_input']})")
        print(f"     → {step['observation'][:80]}")
else:
    print(f"Detail  : {resp.json().get('detail', 'unknown error')}")

## Part 6: Containerisation — Docker

We now have a `Dockerfile` (multi-stage build) and `docker-compose.yml` at the project root.

### Build & Run

```bash
# Build the image
docker build -t personal-ai-api:latest .

# Run it (passing .env file for API keys)
docker run --rm -p 8000:8000 --env-file .env personal-ai-api:latest

# Or use docker-compose (includes ChromaDB volume)
docker compose up --build
```

### What the multi-stage Dockerfile does

| Stage | Base image | Purpose |
|-------|-----------|---------|
| `builder` | `python:3.11-slim` | Installs all wheels into `/opt/venv` |
| `runtime` | `python:3.11-slim` | Copies only the venv — no build tools |

The runtime image runs as a **non-root user** (`appuser`) for security.
ChromaDB data is persisted to a named Docker volume so it survives container restarts.

In [ ]:
import subprocess, sys

# Verify Docker is installed before attempting a build
result = subprocess.run(["docker", "--version"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✅ {result.stdout.strip()}")
    print("\nTo build the image run this in a terminal:")
    print(f"  cd {project_root}")
    print("  docker build -t personal-ai-api:latest .")
    print("\nTo start with docker-compose:")
    print(f"  cd {project_root}")
    print("  docker compose up --build")
else:
    print("⚠️  Docker is not installed or not running.")
    print("   Install Docker Desktop: https://www.docker.com/products/docker-desktop/")

In [ ]:
# ── Teardown: stop the background server ──────────────────────────────────────
# Run this cell when you're done testing.
try:
    server_proc.terminate()
    server_proc.wait(timeout=5)
    print("✅ Server stopped")
except NameError:
    print("ℹ️  server_proc not defined — server was never started")
except Exception as exc:
    print(f"⚠️  Could not stop server: {exc}")

## Day 9 Summary ✅

### What we built today

| Component | Location | Description |
|-----------|----------|-------------|
| LLM factory | `src/utils/llm.py` | Cached `ChatOpenAI` + `OpenAIEmbeddings` pointing to OpenRouter |
| Chat chain | `src/chains/chat.py` | Stateless conversational LCEL chain with slot for message history |
| RAG chain | `src/chains/rag.py` | Chroma retriever → LLM with cited sources |
| Custom tools | `src/tools/custom_tools.py` | calculator, word_counter, get_weather, text_summarizer |
| ReAct agent | `src/agents/react_agent.py` | Tool-calling `AgentExecutor` |
| Vector store | `src/memory/vector_store.py` | build / load / retriever helpers for ChromaDB |
| FastAPI app | `src/api/main.py` | 5 endpoints + SSE streaming + global error handling |
| Schemas | `src/api/models.py` | Pydantic v2 request/response models |
| DI | `src/api/dependencies.py` | `lru_cache` singletons for all shared resources |
| Dockerfile | `Dockerfile` | Multi-stage build → non-root runtime image |
| Compose | `docker-compose.yml` | App + persistent ChromaDB volume |

### Key production patterns learned

1. **Module packaging** — moving notebook code into importable `src/` packages
2. **Dependency injection** — `Depends()` + `lru_cache` keeps heavy objects alive across requests
3. **Async execution** — `run_in_executor` wraps sync LangChain calls so the FastAPI event loop stays responsive
4. **Streaming** — `StreamingResponse` + async generator outputs SSE tokens in real-time
5. **Error handling** — per-endpoint `HTTPException` + global fallback handler
6. **Containerisation** — multi-stage Docker build, non-root user, named volumes for persistence

---

## Next Up: Day 10 — AWS Deployment with CDK

- Define ECS Fargate task + service with AWS CDK in Python
- Push Docker image to ECR
- Wire up an Application Load Balancer with HTTPS
- Manage secrets (API keys) via AWS Secrets Manager
- Set up CloudWatch Logs and LangSmith tracing in production